In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("HomeLoanApprovalSystem").getOrCreate()

Application_schema = StructType([ \
    StructField("CustomerName", StringType(), True), \
    StructField("DOB", StringType(), True), \
    StructField("UIN", StringType(), True), \
    StructField("MailID", StringType(), True), \
    StructField("PhoneNumber", LongType(), True), \
    StructField("City", StringType(), True), \
    StructField("State", StringType(), True), \
    StructField("LivingStatus", StringType(), True), \
    StructField("PinCode", StringType(), True), \
    StructField("LoanAmount", LongType(), True)])

Application_Data_Df=spark.read.format("csv").schema(Application_schema).option("header","true").option("inferSchema","true").load("/Volumes/pyspark_project/spark/datafiles/LoanDataFiles/HomeLoanApplicationData.csv")

Refrance_schema = StructType([ \
    StructField("CustomerName", StringType(), True), \
    StructField("DOB", StringType(), True), \
    StructField("UIN", StringType(), True), \
    #StructField("MailID", StringType(), True), \
    #StructField("PhoneNumber", LongType(), True), \
    StructField("City", StringType(), True), \
    StructField("State", StringType(), True), \
   #StructField("LivingStatus", StringType(), True), \
    StructField("PinCode", StringType(), True), \
    #StructField("LoanAmount", LongType(), True)])
    StructField("CibilScore", LongType(), True),
    StructField("DefaulterFlag", StringType(), True)])

ClientReferenceDataset_Df=spark.read.format("csv").schema(Refrance_schema).option("header","true").option("inferSchema","true").load("/Volumes/pyspark_project/spark/datafiles/LoanDataFiles/ClientReferenceDataset.csv")

display(Application_Data_Df)
display(ClientReferenceDataset_Df)

#Here are the DataFrames in the session:


In [0]:
# Clean and standardize the Home Loan Application DataFrame:
#  - Trim leading/trailing whitespace from UIN and CustomerName for consistent joins
#  - Normalize LivingStatus to uppercase so values like "own" and "Own" match
Application_Data_Df = (
    Application_Data_Df
    .withColumn("UIN", trim(col("UIN")))
    .withColumn("CustomerName", trim(col("CustomerName")))
    .withColumn("LivingStatus", upper(trim(col("LivingStatus"))))
)

# Clean and standardize the Client Reference Dataset DataFrame:
#  - Trim UIN and CustomerName so they align with the Application DataFrame for joining
#  - Normalize DefaulterFlag to uppercase (e.g., "yes" -> "YES") for reliable filtering
ClientReferenceDataset_Df = (ClientReferenceDataset_Df
    .withColumn("UIN", trim(col("UIN")))
    .withColumn("CustomerName", trim(col("CustomerName")))
    .withColumn("DefaulterFlag", upper(trim(col("DefaulterFlag"))))
)

print("Application Records:",Application_Data_Df.count())
print("ClientReferenceDataset Records:",ClientReferenceDataset_Df.count())

Application_Data_Df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in Application_Data_Df.columns
]).show()

ClientReferenceDataset_Df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ClientReferenceDataset_Df.columns
]).show()


display(Application_Data_Df)
display(ClientReferenceDataset_Df)
# Join the two DataFrames on UIN and CustomerName
#  - This is a "semi
     



In [0]:
joined_df = Application_Data_Df.join(ClientReferenceDataset_Df, ["UIN"], "inner")
display(joined_df)
# Filter the joined DataFrame to include only records where the customer is not a defaulter